# [TIL] FastAPI API 테스트 자동화 — pytest와 TestClient로 CRUD 검증하기

**2026.11.05 (목)**

## 1. 오늘 배운 내용

오늘은 FastAPI와 SQLAlchemy ORM을 활용한 백엔드 개발 학습을 이어갔다.

이전까지는 FastAPI에서 API를 구현하고 Swagger UI를 통해 직접 요청을 보내며 정상적으로 작동하는지 확인했다.

오늘은 한 단계 더 나아가 **pytest와 FastAPI의 TestClient를 활용해 API가 올바르게 작동하는지 자동으로 검증하는 방법**을 학습했다.

단순히 API를 만드는 것에서 끝나는 것이 아니라, 작성한 코드가 의도대로 동작하는지 테스트하는 과정도 백엔드 개발의 중요한 부분이라는 것을 알게 되었다.

---

## 2. pytest와 TestClient 이해하기

### pytest란?

pytest는 Python에서 사용하는 테스트 프레임워크다.

함수를 실행한 결과가 예상한 값과 일치하는지 자동으로 확인할 수 있다.

```python
def test_add():
    result = 1 + 2
    assert result == 3
```

여기서 `assert`는 조건이 참인지 확인하는 역할을 한다.

- 조건이 참이면 테스트 통과
- 조건이 거짓이면 테스트 실패

### TestClient란?

FastAPI의 TestClient는 실제 서버를 별도로 실행하지 않고도 API에 HTTP 요청을 보내 테스트할 수 있는 도구다.

```python
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)
```

Swagger UI에서는 사람이 직접 버튼을 눌러 API를 테스트했다면, TestClient를 사용하면 Python 코드로 요청과 검증 과정을 자동화할 수 있다.

## 3. 테스트 환경 구성하기

테스트에 필요한 패키지를 설치했다.

```bash
pip install pytest httpx
```

실습에서 사용한 프로젝트 구조는 다음과 같다.

```text
chapter13/
├── app/
│   ├── main.py
│   ├── db.py
│   ├── models.py
│   ├── schemas.py
│   ├── routers/
│   ├── services/
│   └── repositories/
└── tests/
    └── test_members.py
```

애플리케이션의 기능별 코드를 구분하고, 테스트 코드는 `tests` 폴더에서 별도로 관리하는 구조다.

특히 Router, Service, Repository를 구분하면 요청 처리, 비즈니스 로직, 데이터베이스 접근 역할을 나눌 수 있다는 점도 복습했다.

---

## 4. GET API 테스트하기

가장 기본적인 회원 목록 조회 API부터 테스트했다.

```python
def test_get_members_returns_200():
    response = client.get("/members")

    assert response.status_code == 200
```

코드를 살펴보면 다음과 같다.

**① `client.get("/members")`**

회원 목록을 조회하는 GET 요청을 보낸다.

**② `response.status_code`**

API가 반환한 HTTP 상태 코드를 확인한다.

**③ `assert response.status_code == 200`**

정상적으로 처리되었음을 의미하는 200 OK 응답인지 검증한다.

이처럼 테스트 코드를 작성하면 매번 Swagger UI에서 직접 요청하지 않아도 API의 정상 작동 여부를 확인할 수 있다.

## 5. CRUD API 테스트에서 확인할 내용

CRUD는 데이터베이스의 기본적인 네 가지 기능이다.

| 기능 | HTTP 메서드 | 주요 검증 내용 |
|---|---|---|
| Create | POST | 회원 등록, 201 응답 |
| Read | GET | 목록 및 특정 회원 조회, 200 응답 |
| Update | PUT/PATCH | 회원 정보 수정 결과 |
| Delete | DELETE | 회원 삭제 결과 |

또한 존재하지 않는 회원을 조회하거나 잘못된 데이터를 전송하는 상황처럼 정상적인 요청이 아닌 경우도 테스트할 수 있다.

예를 들어 존재하지 않는 회원을 조회했을 때는 다음과 같은 결과를 기대할 수 있다.

```python
def test_member_not_found():
    response = client.get("/members/999999")

    assert response.status_code == 404
```

단, 테스트 대상 API가 해당 경로와 404 처리를 구현한 경우에 해당한다.

**정상적인 상황뿐 아니라 오류가 발생하는 상황까지 검증하는 것이 중요하다.**

---

## 6. 테스트 자동화의 장점

오늘 학습하면서 이해한 가장 큰 장점은 반복적인 확인 작업을 줄일 수 있다는 것이다.

기존에는 API 코드를 수정할 때마다 Swagger UI에 접속해 직접 요청을 보내야 했다.

하지만 테스트 코드를 작성해 두면 다음 명령어로 여러 테스트를 한 번에 실행할 수 있다.

```bash
python -m pytest -v
```

테스트 자동화의 장점은 다음과 같다.

1. 기존 기능이 정상 작동하는지 빠르게 확인할 수 있다.
2. 코드를 수정한 이후 발생하는 오류를 발견하기 쉽다.
3. 동일한 테스트를 반복해서 실행할 수 있다.
4. 새로운 기능을 추가할 때 기존 기능에 미친 영향을 검증할 수 있다.
5. 프로젝트 규모가 커질수록 유지보수에 도움이 된다.

여기서 기존에 정상 작동하던 기능이 코드 변경 이후에도 여전히 정상적으로 작동하는지 확인하는 것을 **회귀 테스트(Regression Test)**라고 한다.

---

## 7. 오늘 새롭게 이해한 개념

| 개념 | 설명 |
|---|---|
| pytest | Python 테스트 프레임워크 |
| TestClient | FastAPI API 요청 테스트 도구 |
| assert | 예상 결과와 실제 결과 검증 |
| status_code | HTTP 응답 상태 코드 |
| 200 OK | 요청 정상 처리 |
| 201 Created | 리소스 생성 성공 |
| 404 Not Found | 요청한 리소스를 찾을 수 없음 |
| 회귀 테스트 | 코드 수정 후 기존 기능의 정상 동작 확인 |
| 테스트 자동화 | 미리 작성한 테스트를 반복 실행하는 방식 |

## 8. 오늘의 회고

FastAPI를 처음 배울 때는 GET과 POST 요청을 구현하고 Swagger UI에서 정상 응답이 나오는 것만으로도 신기했다.

이후 PostgreSQL과 SQLAlchemy ORM을 연결하면서 실제 데이터베이스를 사용하는 백엔드 구조를 조금씩 이해하게 되었다.

오늘은 여기서 더 나아가 내가 만든 API를 자동으로 테스트하는 방법을 배웠다.

특히 기억에 남는 부분은 **프로그램이 실행된다는 것과 프로그램이 올바르게 작동한다는 것은 다르다**는 점이다.

단순히 오류 없이 실행되는 코드보다, 예상한 결과를 반환하는지 검증할 수 있는 코드가 더 신뢰할 수 있다는 것을 알게 되었다.

아직 테스트 코드를 작성하는 것이 익숙하지는 않지만, 앞으로 프로젝트를 진행하면서 기능 구현과 테스트를 함께 생각하는 습관을 만들어야겠다.

## 9. 다음 학습 목표

- pytest의 `fixture` 개념 이해하기
- 테스트용 데이터베이스와 실제 데이터베이스 분리하기
- 정상 요청뿐 아니라 예외 상황 테스트하기
- API 수정 이후 회귀 테스트 실행하기
- 실제 팀 프로젝트에 테스트 자동화 적용하기

**오늘의 한 줄 정리**

> API를 구현하는 것이 기능을 만드는 과정이라면, 테스트 코드를 작성하는 것은 그 기능을 믿고 사용할 수 있는지 확인하는 과정이다.

#TIL #FastAPI #pytest #SQLAlchemy #백엔드개발